In [ ]:
# Imports and configuration of the corpus
import json
import os
import shutil
import string
import nltk
from nltk.corpus import gutenberg, wordnet
from nltk.wsd import lesk
from openai import OpenAI
from pathlib import Path
from collections import Counter, defaultdict

stemmer = nltk.LancasterStemmer()
TARGET = "bear"

# 1. Connect to your LM Studio local server
API_BASE = "http://localhost:1235/v1"
API_KEY = "local-model"

# 2. Define your target model and the SINGLE target file
MODEL = "google/gemma-4-12b-qat"

# This file will act as both your input and output for continuous patching
# ENTER YOUR bear_bait.json PATH HERE PLEASE
DATASET = "/your-path/bear_bait.json"
DATASET = Path(DATASET)

RUN_API = True                              # False = skip the Gemma queries
MAX_FAILURES = 5                            # stop if the server keeps failing

# The exact 9 valid ontology categories
CATEGORIES = [
    "n_ANIMAL", "n_AGENT",  "n_GEOGRAPHY", 
    "v_CARRY_PHYSICALLY", "v_ENDURE", "v_GIVE_BIRTH", "v_SYMBOLICALLY", 
    "n_COMPOUND", "v_IDIOM",
]

BEAR_HUNT = (
    "You are a strict data classification tool for computational linguistics. Read the sentence and classify "
    "the specific semantic meaning of the target word 'bear' into exactly one of these POS-tagged categories:\n\n"

    "1. n_ANIMAL: (Noun) Refers literally to the large, heavy mammal of the family Ursidae. This category strictly applies when the word represents the biological creature itself, whether in the wild, captivity, or as a general species reference. (e.g., 'The brown bear hunted salmon').\n"

    "2. n_AGENT: (Noun) Represents a person, entity, or mechanical part that performs an action, carries a burden, or transmits information. It typically appears as a suffix in compound nouns indicating someone who transports an item or message. (e.g., 'He was a pall-bearer', 'a tale-bearer').\n"

    "3. n_GEOGRAPHY: (Noun/Verb) Relates to spatial orientation, navigational direction, compass headings, or geographical positioning. It applies when determining a path of travel, an angle, or figuring out one's location in physical space. (e.g., 'The ship will bear north', 'get my bearings').\n\n"

    "4. v_CARRY_PHYSICALLY: (Verb) The act of physically holding, supporting, transporting, or moving a tangible object, weight, or load from one place to another. This requires a literal, physical burden being supported by a person, structure, or vehicle. (e.g., 'The pillars bear the weight', 'bear a heavy box').\n"

    "5. v_ENDURE: (Verb) To emotionally or physically suffer, put up with, withstand, or tolerate a difficult situation, hardship, pain, or annoyance. It involves surviving or managing adversity without giving up. (e.g., 'I cannot bear the terrible pain any longer', 'bear the tragic loss').\n"

    "6. v_GIVE_BIRTH: (Verb) To yield, produce, or bring forth offspring, fruit, or crops. This applies to literal biological reproduction by humans and animals, as well as agricultural yields from trees, plants, or fields. (e.g., 'The apple tree will bear fruit', 'she is to bear a child').\n"

    "7. v_SYMBOLICALLY: (Verb) To possess, exhibit, or hold an abstract trait, title, mark, name, resemblance, or signature. It is a non-physical possession or display of characteristics, rights, or visual similarities rather than carrying a physical object. (e.g., 'He bears the title of King', 'bears a striking resemblance').\n"

    "8. n_COMPOUND: (Noun) Functions as an inseparable part of a capitalized proper noun, formalized geographic location, official institution name, or specific entity title. The word loses its independent meaning to become part of a larger formalized name. (e.g., 'They camped at Bear Mountain').\n"

    "9. v_IDIOM: (Verb) Used as an inseparable component of a fixed idiomatic phrase, phrasal verb, or set expression where the meaning cannot be deduced literally from the individual words alone. (e.g., 'Please bear in mind', 'bring pressure to bear', 'bear with me').\n"

    "Return ONLY the exact category name (e.g., v_CARRY_PHYSICALLY). Do not include quotes, markdown, or conversational text.\n\n"
)

def save_document(data_to_save, filepath):
    """Helper function to cleanly write the JSON array back to disk in the strict 3-line format."""
    filepath = Path(filepath)
    tmp_path = filepath.with_name(filepath.name + ".tmp")
    with open(tmp_path, "w", encoding="utf-8") as f:
        f.write("[\n")
        for i, item in enumerate(data_to_save):
            metadata = {k: v for k, v in item.items() if k not in ["words", "tags"]}
            meta_str = json.dumps(metadata, ensure_ascii=False)
            line1 = "  { " + meta_str[1:-1] + ","
            line2 = '    "words": ' + json.dumps(item.get("words", []), ensure_ascii=False) + ","
            line3 = '    "tags": ' + json.dumps(item.get("tags", []), ensure_ascii=False) + " }"

            f.write(f"{line1}\n{line2}\n{line3}")
            if i < len(data_to_save) - 1:
                f.write(",\n")
            else:
                f.write("\n")
        f.write("]\n")
    os.replace(tmp_path, filepath)


def load_dataset(filepath=DATASET):
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(f"{filepath} not found - run Cell 1 first.")
    return json.loads(filepath.read_text(encoding="utf-8"))


# Process only if category is empty ("") or starts with "ERROR"
def needs_category(record):
    return record.get("category", "") not in CATEGORIES


def sentence_record(record):
    return " ".join([str(w) for w in record.get("words", []) if w])


def find_target(words):
    for i, w in enumerate(words):
        if stemmer.stem(str(w).lower()) == TARGET:
            return i
    return None


def tag_words(words):
    return [tag for _, tag in nltk.pos_tag(words)]


# ============================================================
# PASS 1: CORPUS EXTRACTION (CACHED)
# ============================================================
NLTK_RESOURCES = {
    "gutenberg": "corpora/gutenberg",
    "wordnet": "corpora/wordnet",
    "punkt_tab": "tokenizers/punkt_tab",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
}
for package, resource_path in NLTK_RESOURCES.items():
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package, quiet=True)

# Prevents re-downloading and re-extracting if already in memory
if 'extracted_corpus' not in globals() or len(extracted_corpus) == 0:
    extracted_corpus = []
    for fileid in gutenberg.fileids():
        book_name = fileid.replace(".txt", "")
        for sent_words in gutenberg.sents(fileid):
            if any(stemmer.stem(w.lower()) == TARGET for w in sent_words):
                extracted_corpus.append((book_name, sent_words))
print("Corpus ready.\n")

# 3. Load the current state of the document
existing = {}
if DATASET.exists():
    try:
        raw_rows = json.loads(DATASET.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise RuntimeError(
            f"{DATASET} exists but is not valid JSON ({e}). It was NOT overwritten - "
            "fix it or move it aside, then re-run this cell."
        )
    existing = {str(r.get("id")): r for r in raw_rows}

records = []
audit = Counter()

# 4. Loop through the dataset
for i, (book_name, words) in enumerate(extracted_corpus, start=1):
    rid = f"{i:05d}"
    old = existing.get(rid)

    if old is None:
        audit["rows added (missing from file)"] += 1
        records.append({"id": rid, "book": book_name, "category": "", "human_category": "", "words": words, "tags": tag_words(words)})
    elif old.get("words") != words:
        audit["rows rebuilt (sentence differs from NLTK)"] += 1
        records.append({"id": rid, "book": book_name, "category": "", "human_category": "", "words": words, "tags": tag_words(words)})
    else:
        rec = dict(old)
        fixed = False
        if rec.get("book") != book_name:
            rec["book"] = book_name
            fixed = True
        if not isinstance(rec.get("tags"), list) or len(rec["tags"]) != len(words):
            rec["tags"] = tag_words(words)
            fixed = True
        if "category" not in rec:
            rec["category"] = ""
            fixed = True
        if "human_category" not in rec:
            rec["human_category"] = ""
            audit["rows given a human_category slot"] += 1
        if fixed:
            audit["rows repaired (missing book/tags/category field)"] += 1
        records.append(rec)

extra_ids = set(existing) - {r["id"] for r in records}
if extra_ids:
    audit["rows in file with no NLTK match (dropped)"] = len(extra_ids)

if DATASET.exists() and (audit["rows rebuilt (sentence differs from NLTK)"] or extra_ids):
    backup_path = DATASET.with_name(DATASET.stem + ".backup" + DATASET.suffix)
    shutil.copy2(DATASET, backup_path)
    print(f"Existing rows did not all match NLTK - copy of the old file saved to {backup_path.name}")

if audit or not DATASET.exists():
    save_document(records, DATASET)

print(f"{len(records)} rows in {DATASET.name}")
if audit:
    for label, n in audit.items():
        print(f"{n}  {label}")
else:
    print("All rows loaded and matched with the NLTK data.")

n_missing = sum(needs_category(r) for r in records)
print("All records have a category." if not n_missing else f"{n_missing} of {len(records)} records need a category.\n")

def kill_bear(records):
    # SKIP logic: If it already has a valid category, move to the next one instantly
    todo = [i for i, r in enumerate(records) if needs_category(r)]
    if not todo:
        return
    if not RUN_API:
        print(f"RUN_API is False - skipping the Gemma queries ({len(todo)} records need a category).")
        return

    client = OpenAI(base_url=API_BASE, api_key=API_KEY)
    try:
        client.with_options(timeout=5, max_retries=0).models.list()
    except Exception as e:
        print(f"Could not reach the LLM server at {API_BASE} - skipping the Gemma queries.")
        return

    print(f"Asking {MODEL} to categorize {len(todo)} records...\n")
    consecutive_failures = 0
    for i in todo:
        record = records[i]
        sentence = sentence_record(record)
        try:
            # 5. Prompt Gemma 12B using ONLY the 'user' role with expanded definitions
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": BEAR_HUNT + f"Sentence to classify: {sentence}"}],
                temperature=0.1,   # Bumped slightly to route around deadlocks 
                max_tokens=1024
            )

            raw_result = response.choices[0].message.content or ""

            # 6. Fallback extraction logic (Searches ENTIRE string including <think> tags)
            found_categories = [cat for cat in CATEGORIES if cat.upper() in raw_result.upper()]

            if len(found_categories) == 1:
                category_result = found_categories[0]
            elif len(found_categories) > 1:
                category_result = found_categories[-1]
            else:
                category_result = "ERROR_NO_MATCH"
                print(f"  [DEBUG] Model actually output: {repr(raw_result)}")

            record["category"] = category_result
            consecutive_failures = 0
            print(f"[{i + 1}/{len(records)}] PATCHED: {category_result.ljust(20)} <- '{sentence[:60]}...'")

        except Exception as e:
            record["category"] = "ERROR_API"
            consecutive_failures += 1
            print(f"[{i + 1}/{len(records)}] FAILED: {e}")

        # 7. Checkpoint: Save the document IMMEDIATELY after patching each missing row
        save_document(records, DATASET)

        if consecutive_failures >= MAX_FAILURES:
            print(f"\n{MAX_FAILURES} API failures in a row - stopping.")
            break

    print(f"\nContinuous patching run complete! All updates saved to:\n{DATASET}\n")


kill_bear(records)

missing = [f"[{idx}] {sentence_record(r)}" for idx, r in enumerate(records, start=1) if needs_category(r)]

if missing:
    print(f"{len(missing)} errors remaining:\n")
    print("\n".join(missing))

Corpus ready.

782 rows in bear_bait.json
All rows loaded and matched with the NLTK data.
All records have a category.


In [ ]:
# Manual overrides for where AI calling failed to produce a result
"""

Cell 1 ends by listing every record that still has no valid category, as "[bear_id] sentence".
Give that list to your preferred LLM (I used Gemini, to stay close to the Gemma responses) and ask it to
return the answers in the missing_clues format below, using the category names from CATEGORIES:

    missing_clues = {
        14: "v_ENDURE",
        19: "v_SYMBOLICALLY",
    }

The key is the [bear_id] number from Cell 1's list. Only records that are still missing a valid category get
patched, so it is safe to leave earlier entries in the dictionary.
"""

# Hard-coded classifications for the records the API pass could not categorize
missing_clues = {
    14: "v_ENDURE",
    19: "v_SYMBOLICALLY",
    21: "v_IDIOM",
    42: "v_ENDURE",
    53: "v_SYMBOLICALLY",
    56: "v_SYMBOLICALLY",
    57: "v_IDIOM",
    60: "v_IDIOM",
    66: "v_ENDURE",
    74: "v_ENDURE",
    82: "v_IDIOM",
    83: "v_IDIOM",
    95: "v_ENDURE",
    99: "v_IDIOM",
    112: "v_ENDURE",
    120: "v_GIVE_BIRTH",
    122: "v_GIVE_BIRTH",
    123: "v_ENDURE",
    125: "v_SYMBOLICALLY",
    126: "v_SYMBOLICALLY",
    128: "v_SYMBOLICALLY",
    129: "v_IDIOM",
    130: "v_CARRY_PHYSICALLY",
    133: "v_SYMBOLICALLY",
    134: "v_SYMBOLICALLY",
    143: "v_SYMBOLICALLY",
    144: "v_SYMBOLICALLY",
    145: "v_SYMBOLICALLY",
    146: "v_SYMBOLICALLY",
    148: "v_SYMBOLICALLY",
    149: "v_SYMBOLICALLY",
    150: "v_SYMBOLICALLY",
    151: "v_IDIOM",
    152: "v_SYMBOLICALLY",
    153: "v_SYMBOLICALLY",
    154: "v_SYMBOLICALLY",
    155: "v_SYMBOLICALLY",
    156: "v_SYMBOLICALLY",
    157: "v_SYMBOLICALLY",
    161: "v_SYMBOLICALLY",
    163: "v_SYMBOLICALLY",
    167: "v_SYMBOLICALLY",
    170: "v_SYMBOLICALLY",
    171: "v_SYMBOLICALLY",
    172: "v_SYMBOLICALLY",
    173: "v_SYMBOLICALLY",
    175: "v_SYMBOLICALLY",
    176: "v_ENDURE",
    179: "v_IDIOM",
    201: "v_CARRY_PHYSICALLY",
    202: "v_CARRY_PHYSICALLY",
    204: "v_IDIOM",
    211: "n_AGENT",
    213: "v_IDIOM",
    230: "v_CARRY_PHYSICALLY",
    233: "v_SYMBOLICALLY",
    236: "v_IDIOM",
    241: "v_ENDURE",
    244: "v_SYMBOLICALLY",
    245: "v_SYMBOLICALLY",
    246: "v_SYMBOLICALLY",
    251: "v_ENDURE",
    252: "v_ENDURE",
    255: "v_SYMBOLICALLY",
    256: "v_SYMBOLICALLY",
    257: "v_SYMBOLICALLY",
    258: "v_SYMBOLICALLY",
    259: "v_ENDURE",
    260: "v_ENDURE",
    261: "v_ENDURE",
    263: "v_SYMBOLICALLY",
    265: "v_ENDURE",
    273: "v_ENDURE",
    281: "v_IDIOM",
    283: "v_IDIOM",
    290: "v_SYMBOLICALLY",
    292: "v_IDIOM",
    295: "v_IDIOM",
    296: "v_IDIOM",
    300: "v_IDIOM",
    303: "v_IDIOM",
    309: "v_IDIOM",
    312: "v_ENDURE",
    313: "v_IDIOM",
    314: "v_IDIOM",
    316: "v_IDIOM",
    317: "v_IDIOM",
    318: "v_IDIOM",
    319: "v_IDIOM",
    320: "v_IDIOM",
    325: "v_SYMBOLICALLY",
    329: "v_IDIOM",
    330: "v_ENDURE",
    331: "v_ENDURE",
    332: "v_ENDURE",
    334: "v_IDIOM",
    336: "v_IDIOM",
    337: "v_SYMBOLICALLY",
    342: "v_IDIOM",
    346: "v_ENDURE",
    350: "v_ENDURE",
    351: "v_GIVE_BIRTH",
    374: "n_COMPOUND",
    391: "n_COMPOUND",
    410: "n_COMPOUND",
    415: "n_COMPOUND",
    423: "n_COMPOUND",
    425: "n_COMPOUND",
    430: "n_COMPOUND",
    435: "n_COMPOUND",
    451: "n_COMPOUND",
    453: "n_COMPOUND",
    499: "n_COMPOUND",
    502: "n_ANIMAL",
    505: "n_COMPOUND",
    516: "n_AGENT",
    517: "n_ANIMAL",
    527: "v_CARRY_PHYSICALLY",
    529: "v_SYMBOLICALLY",
    532: "v_IDIOM",
    535: "v_ENDURE",
    536: "v_IDIOM",
    546: "n_AGENT",
    557: "v_IDIOM",
    565: "v_IDIOM",
    569: "v_IDIOM",
    570: "v_ENDURE",
    572: "v_SYMBOLICALLY",
    574: "v_ENDURE",
    575: "v_SYMBOLICALLY",
    578: "v_IDIOM",
    582: "v_SYMBOLICALLY",
    584: "v_IDIOM",
    587: "v_IDIOM",
    599: "n_AGENT",
    601: "v_SYMBOLICALLY",
    609: "v_IDIOM",
    620: "v_IDIOM",
    623: "v_IDIOM",
    628: "n_AGENT",
    629: "v_IDIOM",
    632: "v_IDIOM",
    634: "v_IDIOM",
    636: "v_SYMBOLICALLY",
    637: "n_AGENT",
    642: "n_GEOGRAPHY",
    643: "v_CARRY_PHYSICALLY",
    646: "v_CARRY_PHYSICALLY",
    647: "v_CARRY_PHYSICALLY",
    649: "v_CARRY_PHYSICALLY",
    653: "v_CARRY_PHYSICALLY",
    654: "v_GIVE_BIRTH",
    657: "v_SYMBOLICALLY",
    663: "v_CARRY_PHYSICALLY",
    664: "v_ENDURE",
    668: "v_SYMBOLICALLY",
    677: "v_IDIOM",
    679: "v_IDIOM",
    680: "v_IDIOM",
    687: "v_SYMBOLICALLY",
    689: "v_IDIOM",
    692: "v_CARRY_PHYSICALLY",
    696: "v_IDIOM",
    697: "v_ENDURE",
    698: "v_CARRY_PHYSICALLY",
    699: "v_ENDURE",
    701: "v_SYMBOLICALLY",
    703: "v_IDIOM",
    710: "v_CARRY_PHYSICALLY",
    715: "v_SYMBOLICALLY",
    720: "v_IDIOM",
    721: "v_IDIOM",
    722: "v_SYMBOLICALLY",
    723: "v_SYMBOLICALLY",
    724: "n_AGENT",
    727: "v_CARRY_PHYSICALLY",
    730: "v_CARRY_PHYSICALLY",
    731: "v_SYMBOLICALLY",
    733: "v_IDIOM",
    734: "v_SYMBOLICALLY",
    736: "v_IDIOM",
    737: "v_SYMBOLICALLY",
    741: "v_SYMBOLICALLY",
    742: "v_CARRY_PHYSICALLY",
    743: "v_CARRY_PHYSICALLY",
    745: "n_ANIMAL",
    746: "n_AGENT",
    748: "v_CARRY_PHYSICALLY",
    749: "v_IDIOM",
    750: "n_AGENT",
    751: "v_CARRY_PHYSICALLY",
    753: "v_CARRY_PHYSICALLY",
    757: "v_IDIOM",
    758: "n_AGENT",
    759: "v_GIVE_BIRTH",
    764: "n_AGENT",
    766: "v_CARRY_PHYSICALLY",
    768: "v_IDIOM",
    770: "v_CARRY_PHYSICALLY",
    772: "v_IDIOM",
    773: "v_CARRY_PHYSICALLY",
    774: "n_AGENT",
    780: "v_IDIOM",
    781: "v_ENDURE",
    782: "v_IDIOM"
}

data = load_dataset()

# Guard against typos in LLM output before anything is written
typos = {bear_id: bear_type for bear_id, bear_type in missing_clues.items() if bear_type not in CATEGORIES}
for bear_id, bear_type in typos.items():
    print(f"[{bear_id}] IGNORED: '{bear_type}' is not one of CATEGORIES")

count = 0
for bear_id, bear_spotted in enumerate(data, start=1):
    # If the record is missing a valid category AND we have a valid manual fix for it
    if needs_category(bear_spotted) and bear_id in missing_clues and bear_id not in typos:
        bear_spotted["category"] = missing_clues[bear_id]
        count += 1
        print(f"[{bear_id}] Hard-patched to: {missing_clues[bear_id]}")

if count:
    save_document(data, DATASET)

bears_still_wild = [bear_id for bear_id, bear_spotted in enumerate(data, start=1) if needs_category(bear_spotted)]
if bears_still_wild:
    print(f"\nSuccessfully hard-coded {count} records.")
    print(f"{len(bears_still_wild)} records still have no valid category: {bears_still_wild}")
    print("Add them to missing_clues (or re-run Cell 1 to retry the LLM pass).")
else:
    print("Dataset is 100% complete.")

Dataset is 100% complete.


In [ ]:
# Evaluate the Wordnet dictionary categories that will be used for Lesk

# Imports, stemmer, and TARGET come from Cell 1
data = load_dataset()

# 1. Every surface form in the sentences that stems to the target lemma
bear_forms = set()
for bear_spotted in data:
    # Scan directly through the flat 'words' array
    for word in bear_spotted["words"]:
        if stemmer.stem(word.lower()) == TARGET:
            bear_forms.add(word.lower())

# 2. Seed the inventory with WordNet senses for every form that fits the stem
noun_bears, verb_bears = {}, {}
for form in sorted(bear_forms):
    for bear in wordnet.synsets(form, pos=wordnet.NOUN):
        noun_bears.setdefault(f"({bear.name()}) {bear.definition()}", 0)
    for bear in wordnet.synsets(form, pos=wordnet.VERB):
        verb_bears.setdefault(f"({bear.name()}) {bear.definition()}", 0)

# 3. Lesk each target token, using its tagger POS, and count the sense
failed_to_catch = {"NOUN": 0, "VERB": 0}
for bear_spotted in data:
    words = bear_spotted["words"]
    tags = bear_spotted["tags"]
    
    # Context is now just a lowered copy of the words array
    context = [w.lower() for w in words]

    # zip() cleanly pairs up your two parallel arrays for iteration
    for word, tag in zip(words, tags):
        word = word.lower()
        if stemmer.stem(word) != TARGET:
            continue
            
        if tag.startswith("NN"):
            species, table, label = 'n', noun_bears, "NOUN"
        elif tag.startswith("VB"):
            species, table, label = 'v', verb_bears, "VERB"
        else:
            continue

        bear = lesk(context, word, species)
        if bear is None:
            failed_to_catch[label] += 1
            continue
            
        bear_key = f"({bear.name()}) {bear.definition()}"
        table[bear_key] = table.get(bear_key, 0) + 1

print(f"These forms were matched: {sorted(bear_forms)}")
print(f"Lesk returned {sum(failed_to_catch.values())} cases with no match")

for label, table in [("NOUN", noun_bears), ("VERB", verb_bears)]:
    print(f"\n{label} ({len(table)} total)")
    for bear_definition, bear_hunt in sorted(table.items(), key=lambda x: -x[1]):
        print(f"  {bear_hunt:4d}  {bear_definition}")

These forms were matched: ['bear', 'beare', 'bearer', 'bearers', 'beares', 'bearing', 'bearings', 'bears']
Lesk returned 54 cases with no match

NOUN (12 total)
   152  (bear.n.02) an investor with a pessimistic market outlook; an investor who expects prices to fall and so sells now in order to buy later at a lower price
    65  (bear.n.01) massive plantigrade carnivorous or omnivorous mammals with long shaggy coats and strong claws
    21  (holder.n.03) the person who is in possession of a check or note or bond or document of title that is endorsed to him or to whoever holds it
    10  (carriage.n.03) characteristic way of bearing one's body
     6  (charge.n.14) heraldry consisting of a design or image depicted on a shield
     5  (bearing.n.02) the direction or path along which something moves or along which it lies
     4  (pallbearer.n.01) one of the mourners carrying the coffin at a funeral
     3  (carrier.n.01) someone whose employment involves carrying something
     2  (beari

In [17]:
# Noun vs Verb comparison - treating AI annotations as empirical clues

failed_hunts = 10    # how many misclassifications to print per method


def bear_track(sent_words, tagged_sentence, bear_id):
    bear_footprint = sent_words[bear_id].lower()
    tracking_prediction = "VERB"
    bear_is_noun = False
    bear_is_verb = False

    # Removed "CC" and "IN" from barriers so the scanner can "see through"
    # prepositional phrases ("at all") and conjunctions ("and") to find the true presiding verb.
    natural_obstructions = {",", ".", ":", ";", "WDT", "WP", "WRB"}

    prev_word, prev_tag = ("", "")
    if bear_id > 0:
        prev_word, prev_tag = tagged_sentence[bear_id-1]
        prev_word = prev_word.lower()

    next_tag, next_word, next2_word = ("", "", "")
    if bear_id < len(tagged_sentence) - 1:
        next_word, next_tag = tagged_sentence[bear_id+1]
        next_word = next_word.lower()
    if bear_id < len(tagged_sentence) - 2:
        next2_word = tagged_sentence[bear_id+2][0].lower()

    nearest_prev_verb_tag, nearest_prev_verb_word = None, None
    for i in range(bear_id - 1, -1, -1):
        bear_trap, bear_bait = tagged_sentence[i]
        if bear_bait in natural_obstructions: break
        if bear_bait.startswith("VB") or bear_bait == "MD":
            nearest_prev_verb_tag = bear_bait
            nearest_prev_verb_word = bear_trap.lower()
            break

    nearest_next_verb_tag = None
    for i in range(bear_id + 1, len(tagged_sentence)):
        _, bear_bait = tagged_sentence[i]
        if bear_bait in natural_obstructions: break
        if bear_bait.startswith("VB") or bear_bait == "MD":
            nearest_next_verb_tag = bear_bait
            break

    # ------------------------------------------------------------
    # RULE A: LEFT-SIDE CONTEXT
    # ------------------------------------------------------------
    if bear_id > 0:
        # 1. The Unbroken Auxiliary Chain
        if nearest_prev_verb_tag == "MD" or nearest_prev_verb_word in {"do", "does", "did", "can", "could", "will", "would", "shall", "should", "may", "might", "must"}:
            if bear_footprint == "bears":
                bear_is_noun = True
            else:
                bear_is_verb = True

        # 2. Coordinate Conjunctions ("know and bear")
        elif prev_tag == "CC" and nearest_prev_verb_tag:
            bear_is_verb = True

        # 3. Infinitive
        elif prev_tag == "TO":
            if bear_footprint == "bears" or sent_words[bear_id].istitle():
                bear_is_noun = True
            else:
                bear_is_verb = True

        # 4. Subject Pronoun
        elif prev_tag == "PRP":
            if bear_footprint == "bears" and prev_word not in {"he", "she", "it"}:
                bear_is_noun = True
            elif next_tag.startswith("NN"):
                bear_is_noun = True
            else:
                bear_is_verb = True

        # 5. Noun Modifiers
        elif prev_tag in {"DT", "PRP$", "POS", "CD", "JJ", "JJR", "JJS", "PDT", "NN", "NNS"}:
            if next_tag.startswith("NN"):
                bear_is_noun = True
            elif prev_tag == "NNS" or (prev_tag == "CD" and prev_word not in {"one", "1", "a", "an"}):
                if bear_footprint == "bear": bear_is_verb = True
                else: bear_is_noun = True
            elif prev_tag == "NN":
                if bear_footprint == "bears": bear_is_verb = True
                else: bear_is_noun = True
            else:
                bear_is_noun = True

        # 6. Preceding Action Verbs (S-V-[O])
        elif nearest_prev_verb_tag:
            bear_is_noun = True

    # ------------------------------------------------------------
    # RULE B: RIGHT-SIDE CONTEXT (Only if A didn't trigger)
    # ------------------------------------------------------------
    if bear_id < len(tagged_sentence) - 1 and not bear_is_verb and not bear_is_noun:
        if next_word == "in" and next2_word == "mind": bear_is_verb = True
        elif next_word == "with" and next2_word in {"me", "us", "you", "him", "her", "them", "it"}: bear_is_verb = True
        elif next_tag == "RP": bear_is_verb = True
        elif next_tag in {"DT", "PRP$", "PDT"}: bear_is_verb = True
        elif nearest_next_verb_tag: bear_is_noun = True
        elif next_tag == "IN": bear_is_noun = True

    # ------------------------------------------------------------
    # RULE C: LEXICAL OVERRIDES
    # ------------------------------------------------------------
    if bear_footprint in {"bearer", "bearers", "bearings"}:
        bear_is_noun = True
        bear_is_verb = False

    if bear_is_noun and not bear_is_verb:
        tracking_prediction = "NOUN"
    return tracking_prediction


data = load_dataset()

scored = []
form_counts = Counter()
skipped_no_forest = 0
skipped_no_bear = 0
multiple_bears_spotted = 0

for rec_idx, bear_spotted in enumerate(data, start=1):
    bear_type = bear_spotted.get("category", "")
    if bear_type not in CATEGORIES:
        skipped_no_forest += 1
        continue

    words = bear_spotted["words"]
    tags = bear_spotted["tags"]
    bear_id = find_target(words)
    if bear_id is None:
        skipped_no_bear += 1
        continue
    if sum(1 for w in words if stemmer.stem(str(w).lower()) == TARGET) > 1:
        multiple_bears_spotted += 1

    clue = "NOUN" if bear_type.startswith("n_") else "VERB"
    tagger_prediction = "NOUN" if tags[bear_id].startswith("NN") else "VERB"
    rules_prediction = bear_track(words, list(zip(words, tags)), bear_id)

    target = words[bear_id].lower()
    form_counts[f"{target} ({'n.' if clue == 'NOUN' else 'v.'})"] += 1
    scored.append({
        "rec": rec_idx,
        "target": target,
        "tag": tags[bear_id],
        "category": bear_type,
        "clue": clue,
        "tagger": tagger_prediction,
        "rules": rules_prediction,
        "context": " ".join(words[max(0, bear_id - 5):min(len(words), bear_id + 6)]),
    })

total = len(scored)
bear_noun = sum(1 for s in scored if s["clue"] == "NOUN")
bear_verb = total - bear_noun


def percentage(numerator, denominator):
    return f"{numerator / denominator * 100:>10.1f}%" if denominator else f"{'N/A':>11}"


def bear_row(bait, trap):
    correct = sum(1 for s in scored if s[trap] == s["clue"])
    noun_bear = sum(1 for s in scored if s["clue"] == "NOUN" and s[trap] == "NOUN")
    verb_bear = sum(1 for s in scored if s["clue"] == "VERB" and s[trap] == "VERB")
    fraction = f"{correct}/{total}"
    print(f"{bait:<14} | {fraction:>11} | {percentage(correct, total)} | {percentage(noun_bear, bear_noun)} | {percentage(verb_bear, bear_verb)}")


# ============================================================
# REPORT OUTPUT
# ============================================================

print(f"{total} sentences with '{TARGET}' ({bear_noun} nouns and {bear_verb} verbs)")
if skipped_no_forest:
    print(f"Skipped, no valid category: {skipped_no_forest}  (run Cell 1 / Cell 2 to complete the dataset)")
if skipped_no_bear:
    print(f"Skipped, no '{TARGET}' token found: {skipped_no_bear}")
print(f"{multiple_bears_spotted} sentences with more than one '{TARGET}' instance \n")

print(f"{'METHOD':<14} | {'CORRECT':>11} | {'TOTAL ACCURACY':>11} | {'NOUN TOTAL':>11} | {'VERB TOTAL':>11}")
print("-" * 72)
bear_row("POS tagger", "tagger")
bear_row("Rules", "rules")
print("-" * 72)

print("\nSurface Nouns:")
for form, n in form_counts.most_common():
    if "(n.)" in form:
        print(f"  {form.replace(' (n.)', ''):<16}{n}")

print("\nSurface Verbs:")
for form, n in form_counts.most_common():
    if "(v.)" in form:
        print(f"  {form.replace(' (v.)', ''):<16}{n}")

782 sentences with 'bear' (224 nouns and 558 verbs)
16 sentences with more than one 'bear' instance 

METHOD         |     CORRECT | TOTAL ACCURACY |  NOUN TOTAL |  VERB TOTAL
------------------------------------------------------------------------
POS tagger     |     701/782 |       89.6% |       95.1% |       87.5%
Rules          |     617/782 |       78.9% |       88.8% |       74.9%
------------------------------------------------------------------------

Surface Nouns:
  bear            176
  bearer          14
  bearers         13
  bears           11
  bearing         5
  bearings        2
  beare           2
  beares          1

Surface Verbs:
  bear            414
  bearing         69
  beare           45
  bears           16
  beares          13
  bearings        1


In [18]:
# Bag of Words Rules
dataset_filename = DATASET   # defined in Cell 1

LESK_TO_BEAR_MAP = {
    "bear.n.01": "n_ANIMAL", 
    "bear.n.02": "n_AGENT", "holder.n.03": "n_AGENT", "pallbearer.n.01": "n_AGENT", "carrier.n.01": "n_AGENT", "bearer.n.02": "n_AGENT",
    "bearing.n.02": "n_GEOGRAPHY", "bearing.n.01": "n_GEOGRAPHY", "bearing.n.03": "n_GEOGRAPHY", 
    "bear.v.04": "v_CARRY_PHYSICALLY", "bear.v.05": "v_CARRY_PHYSICALLY", "hold.v.11": "v_CARRY_PHYSICALLY", "wear.v.02": "v_CARRY_PHYSICALLY", "hold.v.14": "v_CARRY_PHYSICALLY", "bearing.n.06": "v_CARRY_PHYSICALLY",
    "bear.v.06": "v_ENDURE", "digest.v.03": "v_ENDURE", "carriage.n.03": "v_ENDURE", "behave.v.02": "v_ENDURE",
    "give_birth.v.01": "v_GIVE_BIRTH", "have_a_bun_in_the_oven.v.01": "v_GIVE_BIRTH", 
    "bear.v.11": "v_SYMBOLICALLY", "bear.v.01": "v_SYMBOLICALLY", "charge.n.14": "v_SYMBOLICALLY", 
    "yield.v.10": "v_IDIOM"
}

NOUN_BEARS = ["n_ANIMAL", "n_AGENT", "n_GEOGRAPHY", "n_COMPOUND"]
VERB_BEARS = ["v_CARRY_PHYSICALLY", "v_ENDURE", "v_GIVE_BIRTH", "v_SYMBOLICALLY", "v_IDIOM"]

# --- STRICTLY BALANCED KEYWORD WINDOWS (EXACTLY 100 WORDS EACH) ---

FOREST_BEARS = {
    "forest", "woods", "cave", "honey", "claws", "fur", "growl", "beast", "wild", "creature",
    "hunt", "trap", "dog", "animal", "savage", "roar", "muzzle", "paws", "teeth", "shaggy",
    "wilderness", "trees", "den", "track", "trail", "rifle", "gun", "shoot", "hound", "prey",
    "carnivore", "fierce", "predator", "brute", "woodsman", "hunter", "cabin", "mountain", "timber", "thicket",
    "brush", "game", "hide", "pelt", "snarl", "grunt", "lumbering", "massive", "cub", "hibernation",
    "winter", "berries", "stream", "fish", "salmon", "woodland", "branches", "leaves", "pine", "oak",
    "nature", "zoology", "specimen", "captive", "cage", "zoo", "circus", "chained", "tame", "collar",
    "pole", "dancing", "trick", "instinct", "footprints", "clawed", "scratched", "bark", "sniff", "snout",
    "species", "wildcat", "wolf", "cougar", "panther", "elk", "deer", "moose", "flesh", "bone",
    "bloody", "fangs", "mane", "tail", "quadruped", "flora", "fauna", "jungle", "plain", "feral"
}

AGENT_BEARS = {
    "messenger", "letter", "check", "note", "document", "market", "stock", "shares", "wall street", "broker",
    "investor", "finance", "financial", "exchange", "trading", "trader", "sell", "buy", "price", "prices",
    "dispatch", "courier", "errand", "delivery", "post", "mail", "envelope", "parcel", "package", "missive",
    "correspondence", "telegram", "wire", "message", "reply", "response", "bearer", "agent", "representative", "proxy",
    "delegate", "envoy", "ambassador", "emissary", "herald", "harbinger", "tidings", "news", "information", "word",
    "report", "rumor", "gossip", "tale", "story", "portfolio", "briefcase", "satchel", "pouch", "wallet",
    "purse", "coin", "currency", "bill", "draft", "promissory", "signature", "endorsement", "endorsed", "payee",
    "payer", "banker", "bank", "clerk", "teller", "transaction", "commerce", "trade", "merchant", "mercantile",
    "economic", "economy", "slump", "downturn", "pessimistic", "speculator", "dividend", "yield", "investment", "capitalization",
    "equity", "bond", "ledger", "certificate", "deed", "contract", "warrant", "summons", "writ", "subpoena"
}

GEO_BEARS = {
    "north", "south", "east", "west", "compass", "navigate", "navigation", "navigator", "ship", "sail",
    "sailing", "course", "heading", "steer", "steering", "port", "starboard", "bow", "stern", "latitude",
    "longitude", "map", "chart", "sea", "ocean", "voyage", "travel", "path", "direction", "route",
    "way", "bearing", "bearings", "orient", "orientation", "location", "position", "locate", "coordinates", "degrees",
    "angle", "azimuth", "meridian", "equator", "pole", "polar", "magnetic", "needle", "astrolabe", "sextant",
    "horizon", "landmark", "distance", "miles", "leagues", "journey", "expedition", "track", "trail", "wind",
    "current", "tide", "coast", "shore", "cape", "island", "strait", "channel", "gulf", "bay",
    "terrain", "topography", "survey", "surveyor", "point", "destination", "arrival", "departure", "transit", "passage",
    "cross", "crossing", "trek", "wander", "explore", "explorer", "discover", "discovery", "globe", "world",
    "atlas", "spatial", "geometry", "calculate", "plot", "plotted", "trajectory", "vector", "aim", "align"
}

BIRTH_BEARS = {
    "born", "birth", "child", "children", "son", "daughter", "mother", "father", "twins", "offspring",
    "infant", "baby", "babe", "parent", "pregnant", "pregnancy", "womb", "conceived", "conception", "lineage",
    "ancestry", "descendant", "heir", "inherited", "bloodline", "kin", "kindred", "family", "bred", "breeding",
    "generation", "progeny", "delivered", "delivery", "midwife", "nurse", "suckle", "nursemaid", "maternal", "paternal",
    "maternity", "paternity", "newborn", "boy", "girl", "sibling", "brother", "sister", "youthful", "childhood",
    "rearing", "raising", "infancy", "toddler", "youth", "begot", "begotten", "sired", "sire", "matriarch",
    "patriarch", "familial", "relations", "relatives", "cousin", "uncle", "aunt", "nephew", "niece", "grandchild",
    "grandson", "granddaughter", "native", "origins", "blood", "origin", "cradle", "nursery", "pedigree", "stock",
    "race", "breed", "legitimate", "illegitimate", "bastard", "expecting", "confinement", "labor", "pangs", "childbirth",
    "fruit", "seed", "foster", "orphan", "guardian", "ward", "descend", "descended", "ancestor", "forefather"
}

CARRY_BEARS = {
    "weight", "heavy", "load", "basket", "shoulder", "wood", "water", "tray", "coffin", "carry",
    "transport", "move", "bring", "take", "fetch", "convey", "haul", "drag", "pull", "tug",
    "lug", "lift", "heave", "hoist", "elevate", "raise", "hold", "support", "sustain", "prop",
    "pillar", "column", "beam", "arch", "wall", "roof", "floor", "foundation", "base", "structure",
    "back", "arms", "hands", "head", "neck", "carriage", "cart", "wagon", "wheelbarrow", "sled",
    "sledge", "boat", "ship", "vessel", "cargo", "freight", "baggage", "luggage", "trunk", "chest",
    "box", "crate", "sack", "bag", "pouch", "pack", "knapsack", "backpack", "encumbrance", "physically",
    "carried", "lifting", "holding", "gripping", "grasping", "clutching", "timber", "stone", "rock", "iron",
    "steel", "supplies", "provisions", "goods", "wares", "merchandise", "bucket", "pail", "pitcher", "jug",
    "barrel", "cask", "lumber", "bricks", "coal", "harvest", "crop", "grain", "wheat", "corn"
}

SYMBOL_BEARS = {
    "shield", "crest", "arms", "banner", "heraldic", "emblem", "symbol", "sign", "token", "flag",
    "standard", "pennant", "ensign", "colors", "insignia", "badge", "seal", "stamp", "signet", "escutcheon",
    "blazon", "rampant", "passant", "chevron", "knight", "chivalry", "armor", "helmet", "sword", "nobility",
    "aristocrat", "king", "queen", "prince", "crown", "coronet", "sceptre", "throne", "dynasty", "parchment",
    "scroll", "charter", "grant", "patent", "symbolic", "mark", "hallmark", "motto", "monarch", "lord",
    "name", "title", "office", "resemblance", "character", "right", "rights", "privilege", "claim", "authority",
    "power", "jurisdiction", "command", "control", "responsibility", "duty", "role", "position", "rank", "station",
    "status", "degree", "signature", "endorse", "likeness", "similarity", "trace", "relation", "honor", "reputation",
    "fame", "glory", "shame", "disgrace", "stigma", "trait", "feature", "quality", "attribute", "evidence",
    "proof", "testify", "witness", "oath", "official", "legitimate", "valid", "possess", "own", "prerogative"
}

ENDURE_BEARS = {
    "cannot", "hardly", "ill", "pain", "sorrow", "grief", "burden", "trial", "patience", "suffer",
    "stomach", "endure", "tolerate", "stand", "abide", "weather", "withstand", "brave", "confront", "survive",
    "outlast", "undergo", "experience", "feel", "hurt", "ache", "agony", "anguish", "misery", "woe",
    "sadness", "despair", "hardship", "adversity", "affliction", "cross", "penalty", "punishment", "wrath", "anger",
    "fury", "abuse", "insult", "injury", "wrong", "injustice", "offense", "assault", "blow", "shock",
    "strain", "stress", "tension", "pressure", "heat", "cold", "intolerable", "unbearable", "insufferable", "unendurable",
    "patiently", "stoically", "bravely", "loss", "torment", "torture", "hard", "difficult", "heavy", "cruel",
    "proud", "manner", "demeanor", "mien", "deportment", "carriage", "posture", "bearing", "stance", "attitude",
    "aspect", "appearance", "presence", "dignity", "dignified", "poise", "composure", "composed", "calm", "confident",
    "arrogant", "arrogance", "humble", "humility", "bold", "gait", "countenance", "figure", "stature", "physique"
}

# Structure of the rules (points system):
#   NOUN block: n_COMPOUND rules (+10 each); FOREST_BEARS and AGENT_BEARS bags (+1 per cue)
#   VERB block: v_IDIOM rules (+10 each); CARRY_BEARS, ENDURE_BEARS, BIRTH_BEARS bags (+1 per cue)
#   POS-independent block: GEO_BEARS and SYMBOL_BEARS bags (+1 per cue), scored for every token
def classify_rules(bear_sentence, target_bear, bear_pos):
    if not bear_sentence: return "UNCLASSIFIED"
    bear_tokens = bear_sentence.split()
    bear_tokens_lower = [t.lower().strip(string.punctuation) for t in bear_tokens]
    
    target_bear_idx = -1
    for i, t in enumerate(bear_tokens_lower):
        if t == target_bear.lower() or t == target_bear.lower() + "s":
            target_bear_idx = i
            break

    if target_bear_idx == -1: return "UNCLASSIFIED"

    bear_scores = Counter()
    local_forest = set(bear_tokens_lower[max(0, target_bear_idx - 6):min(len(bear_tokens_lower), target_bear_idx + 7)])

    if bear_pos == "NOUN":
        # Compound-noun rules (+10 each)
        if target_bear_idx > 0 and bear_tokens[target_bear_idx].istitle() and bear_tokens[target_bear_idx - 1].istitle():
            bear_scores["n_COMPOUND"] += 10

        if "-" in bear_tokens_lower[target_bear_idx]:
            bear_scores["n_COMPOUND"] += 10

        if len(bear_tokens_lower[target_bear_idx]) > 4 and bear_tokens_lower[target_bear_idx].endswith("bear"):
            if bear_tokens_lower[target_bear_idx] not in {"forbear", "overbear"}:
                bear_scores["n_COMPOUND"] += 10

        if target_bear_idx > 0 and bear_tokens_lower[target_bear_idx - 1] in {"polar", "grizzly", "teddy", "black", "brown"}:
            bear_scores["n_COMPOUND"] += 10

        # Bag-of-words cues (+1 per distinct word in the window)
        bear_scores["n_ANIMAL"] += len(local_forest.intersection(FOREST_BEARS))
        bear_scores["n_AGENT"] += len(local_forest.intersection(AGENT_BEARS))

        bear_scores["n_ANIMAL"] += 0.5   # default when no cue fires

    elif bear_pos == "VERB":
        # Idiom rules (+10 each)
        forest_ahead = bear_tokens_lower[target_bear_idx + 1 : min(len(bear_tokens_lower), target_bear_idx + 4)]
        if "in" in forest_ahead and ("mind" in forest_ahead or "remembrance" in forest_ahead):
            bear_scores["v_IDIOM"] += 10

        if set(forest_ahead).intersection({"fruit", "witness", "brunt", "testimony"}):
            bear_scores["v_IDIOM"] += 10

        if target_bear_idx > 1 and bear_tokens_lower[target_bear_idx - 1] == "and" and bear_tokens_lower[target_bear_idx - 2] in {"grin", "smile"}:
            bear_scores["v_IDIOM"] += 10

        if target_bear_idx < len(bear_tokens_lower) - 1 and bear_tokens_lower[target_bear_idx + 1] in {"down", "away", "out", "up"}:
            bear_scores["v_IDIOM"] += 10

        # Bag-of-words cues (+1 per distinct word in the window)
        bear_scores["v_CARRY_PHYSICALLY"] += len(local_forest.intersection(CARRY_BEARS))
        bear_scores["v_ENDURE"] += len(local_forest.intersection(ENDURE_BEARS))
        bear_scores["v_GIVE_BIRTH"] += len(local_forest.intersection(BIRTH_BEARS))

        bear_scores["v_CARRY_PHYSICALLY"] += 0.5   # default when no cue fires

    # POS-independent categories: same +1 per cue whether the token is a noun or a verb
    bear_scores["n_GEOGRAPHY"] += len(local_forest.intersection(GEO_BEARS))
    bear_scores["v_SYMBOLICALLY"] += len(local_forest.intersection(SYMBOL_BEARS))

    if not bear_scores:
        return "UNCLASSIFIED"
        
    return max(bear_scores, key=bear_scores.get)


if os.path.exists(DATASET):
    with open(DATASET, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    bear_metrics = {bear_type: {"total": 0, "lesk_match": 0, "rule_match": 0} for bear_type in NOUN_BEARS + VERB_BEARS}
    noun_bear_totals = {"total": 0, "lesk_match": 0, "rule_match": 0}
    verb_bear_totals = {"total": 0, "lesk_match": 0, "rule_match": 0}
    grand_bear_totals = {"total": 0, "lesk_match": 0, "rule_match": 0}
    
    for bear_id, bear_spotted in enumerate(data):
        words = bear_spotted.get("words", [])
        tags = bear_spotted.get("tags", [])
        sentence = " ".join([str(w) for w in words if w])
        bear_clue = bear_spotted.get("category", "UNCLASSIFIED")
        
        target = "bear"
        true_bear_pos = "NOUN"
        for w, bear_pos in zip(words, tags):
            if "bear" in str(w).lower():
                target = str(w).lower()
                true_bear_pos = "VERB" if str(bear_pos).startswith("V") else "NOUN"
                break
        
        pos_category = 'n' if true_bear_pos == "NOUN" else 'v'
        
        try:
            bear_sense = lesk(words, target, pos_category)
            bear_lesk = bear_sense.name() if bear_sense else "UNKNOWN"
        except Exception:
            bear_lesk = "UNKNOWN"
            
        lesk_bear_prediction = LESK_TO_BEAR_MAP.get(bear_lesk, "UNCLASSIFIED")
        rule_bear_prediction = classify_rules(sentence, target, true_bear_pos)
        
        if bear_clue in bear_metrics:
            bear_metrics[bear_clue]["total"] += 1
            if lesk_bear_prediction == bear_clue: bear_metrics[bear_clue]["lesk_match"] += 1
            if rule_bear_prediction == bear_clue: bear_metrics[bear_clue]["rule_match"] += 1
            
            if bear_clue in NOUN_BEARS:
                noun_bear_totals["total"] += 1
                if lesk_bear_prediction == bear_clue: noun_bear_totals["lesk_match"] += 1
                if rule_bear_prediction == bear_clue: noun_bear_totals["rule_match"] += 1
            elif bear_clue in VERB_BEARS:
                verb_bear_totals["total"] += 1
                if lesk_bear_prediction == bear_clue: verb_bear_totals["lesk_match"] += 1
                if rule_bear_prediction == bear_clue: verb_bear_totals["rule_match"] += 1
                
            grand_bear_totals["total"] += 1
            if lesk_bear_prediction == bear_clue: grand_bear_totals["lesk_match"] += 1
            if rule_bear_prediction == bear_clue: grand_bear_totals["rule_match"] += 1

    def print_bear_stat_row(label, bear_data_dict):
        total = bear_data_dict["total"]
        if total == 0:
            print(f"{label:<22} | {total:<5} | {'N/A':>8} | {'N/A':>8}")
            return
        lesk_bear_percentage = (bear_data_dict["lesk_match"] / total) * 100
        rule_bear_percentage = (bear_data_dict["rule_match"] / total) * 100
        print(f"{label:<22} | {total:<5} | {lesk_bear_percentage:>7.1f}% | {rule_bear_percentage:>7.1f}%")

    print(f"{'CATEGORY':<22} | {'COUNT':<5} | {'LESK %':>8} | {'RULES %':>8}")
    print("=" * 52)

    for bear_type in NOUN_BEARS:
        print_bear_stat_row(bear_type, bear_metrics[bear_type])
    print("-" * 52)
    print_bear_stat_row("TOTAL NOUNS", noun_bear_totals)
    print("=" * 52)

    for bear_type in VERB_BEARS:
        print_bear_stat_row(bear_type, bear_metrics[bear_type])
    print("-" * 52)
    print_bear_stat_row("TOTAL VERBS", verb_bear_totals)
    print("=" * 52)

    print_bear_stat_row("GRAND TOTAL", grand_bear_totals)
    print("=" * 52)

else:
    print(f"Dataset not found at {dataset_filename}")

CATEGORY               | COUNT |   LESK % |  RULES %
n_ANIMAL               | 46    |    28.3% |    82.6%
n_AGENT                | 27    |   100.0% |    55.6%
n_GEOGRAPHY            | 7     |    28.6% |    71.4%
n_COMPOUND             | 144   |     0.0% |    89.6%
----------------------------------------------------
TOTAL NOUNS            | 224   |    18.8% |    83.5%
v_CARRY_PHYSICALLY     | 109   |     1.8% |    46.8%
v_ENDURE               | 193   |    40.4% |    18.7%
v_GIVE_BIRTH           | 46    |     8.7% |    17.4%
v_SYMBOLICALLY         | 115   |    10.4% |    14.8%
v_IDIOM                | 95    |     0.0% |    31.6%
----------------------------------------------------
TOTAL VERBS            | 558   |    17.2% |    25.4%
GRAND TOTAL            | 782   |    17.6% |    42.1%


In [19]:
# Syntactic rules

dataset_filename = DATASET   # defined in Cell 1
SHOW_RULE_STATS = False           # True = for every rule, how many records it decided and how often it was right
SHOW_BEAR_MISSES = False               # True = where the records of each Gemma category ended up

NOUN_BEARS = ["n_ANIMAL", "n_AGENT", "n_GEOGRAPHY", "n_COMPOUND"]
VERB_BEARS = ["v_CARRY_PHYSICALLY", "v_ENDURE", "v_GIVE_BIRTH", "v_SYMBOLICALLY", "v_IDIOM"]

MODALS = {"can", "cannot", "could", "couldn", "would", "wouldn", "might", "mightn", "must", "mustn"}
NEGATORS = {"not", "never", "ill", "hardly"}
DIRECTIONS = {"north", "south", "east", "west", "left", "right"}
CHILD_WORDS = {"child", "children", "son", "sons", "daughter", "daughters", "twins", "babe", "baby", "infant", "young", "heir", "heirs", "offspring", "seed"}
CLAUSE_END = {",", ".", ";", ":", "!", "?"}
ARCHAIC_WORDS = {"thou", "thee", "thy", "thine", "ye", "hath", "hast", "doth", "dost", "shalt", "wilt", "canst", "wouldst", "unto", "wherefore", "behold", "saith", "verily", "yea", "nay", "ere", "hither", "thither", "whence", "methinks"}
LOAD_WORDS = {"weight", "load", "loads", "box", "boxes", "basket", "baskets", "coffin", "pitcher", "bucket", "buckets", "shoulder", "shoulders", "bundle", "bundles", "sack", "sacks", "stone", "stones", "litter", "ark"}
PLACE_PREPS = {"on", "upon", "along", "across", "over", "through", "into", "from"}
OBJECT_TAGS = {"PRP", "DT", "PRP$", "CD", "JJ", "NN", "NNS", "NNP"}
SYMBOL_WORDS = {"name", "names", "title", "mark", "marks", "resemblance", "likeness", "signature", "trait", "traits", "stamp", "trace", "traces", "inscription", "relation"}


def classify_bear_syntax(words, tags, bear_id, bear_pos):
    """Decision list: the first rule whose condition is true decides. Returns (category, rule label)."""
    token_count = len(words)
    bear_tokens = [str(w) for w in words]
    bear_low = [t.lower() for t in bear_tokens]
    bear_tokens_lower = [t.strip(string.punctuation) for t in bear_low]

    def W(k): return bear_low[k] if 0 <= k < token_count else ""
    def T(k): return tags[k] if 0 <= k < token_count else ""

    bear_form = bear_low[bear_id]

    # a word after a modal or "to" is a verb, skipping not/never/ill/ever ("can not" = "cannot" = "can't" = "can")
    scan_idx = bear_id - 1
    while scan_idx >= 0 and bear_low[scan_idx] in NEGATORS | {"ever", "'", "t"}:
        scan_idx -= 1
    if bear_pos == "NOUN" and (W(scan_idx) in MODALS or W(scan_idx) == "to") and bear_tokens[bear_id].islower():
        bear_pos = "VERB"

    # ---- compound nouns (same as Cell 5)
    title_title = bear_id > 0 and bear_tokens[bear_id].istitle() and bear_tokens[bear_id - 1].istitle()
    hyphenated = "-" in bear_tokens_lower[bear_id]
    bear_suffix = len(bear_tokens_lower[bear_id]) > 4 and bear_tokens_lower[bear_id].endswith("bear") and bear_tokens_lower[bear_id] not in {"forbear", "overbear"}
    bear_adjective = bear_id > 0 and bear_tokens_lower[bear_id - 1] in {"polar", "grizzly", "teddy", "black", "brown"}

    # ---- idioms (same as Cell 5)
    window_ahead = bear_tokens_lower[bear_id + 1 : min(token_count, bear_id + 4)]
    bear_in_mind = "in" in window_ahead and ("mind" in window_ahead or "remembrance" in window_ahead)
    bear_fruit = bool(set(window_ahead).intersection({"fruit", "witness", "brunt", "testimony"}))
    grin_and_bear = bear_id > 1 and bear_tokens_lower[bear_id - 1] == "and" and bear_tokens_lower[bear_id - 2] in {"grin", "smile"}
    bear_particle = bear_id < token_count - 1 and bear_tokens_lower[bear_id + 1] in {"down", "away", "out", "up"}

    # ---- animal: bear(s) after an article or possessive, adjectives allowed in between
    adj_scan_idx = bear_id - 1
    while T(adj_scan_idx) in {"JJ", "JJR", "JJS"}:
        adj_scan_idx -= 1
    article_bear = bear_form in {"bear", "bears"} and T(adj_scan_idx) in {"DT", "PRP$"}                     # the bear, a great shaggy bear

    # ---- birth, symbolic, geography
    child_next = any(W(k) in CHILD_WORDS for k in range(bear_id + 1, bear_id + 4))                      # bear a child
    symbol_next = any(W(k) in SYMBOL_WORDS for k in range(bear_id + 1, bear_id + 4))                    # bear the name of
    resemblance = W(bear_id + 1) in {"a", "an"} and "to" in [W(bear_id + 2), W(bear_id + 3), W(bear_id + 4)]        # bear a resemblance to
    archaic = any(w in ARCHAIC_WORDS for w in bear_low)                                          # thou, thee, hath, unto...
    direction_next = W(bear_id + 1) in DIRECTIONS or (W(bear_id + 1) == "to" and W(bear_id + 2) == "the" and W(bear_id + 3) in DIRECTIONS)   # bear north

    # ---- endure
    modal_before = any(W(k) in MODALS | NEGATORS for k in range(bear_id - 3, bear_id))                 # could not bear
    infinitive_next = W(bear_id + 1) == "to" and T(bear_id + 2) not in {"DT", "PRP$", "CD", "JJ", "NNP", "NNPS"}   # bear to see
    ing_next = T(bear_id + 1) == "VBG"                                                            # bear seeing
    clause_end_next = W(bear_id + 1) in CLAUSE_END                                                # more than she could bear.
    pronoun_next = W(bear_id + 1) in {"it", "this", "that"}                                       # bear it

    # ---- carry: a load word follows, or a noun phrase followed by a place word (before any punctuation)
    clause = []
    for w in bear_low[bear_id + 1 : bear_id + 7]:
        if w in CLAUSE_END:
            break
        clause.append(w)
    load_next = any(W(k) in LOAD_WORDS for k in range(bear_id + 1, bear_id + 4))                        # bear the weight
    object_then_place = T(bear_id + 1) in OBJECT_TAGS and any(w in PLACE_PREPS for w in clause[1:])   # bear it on his back

    if bear_pos == "NOUN":
        # compound
        if title_title:                              return "n_COMPOUND", "C1 Title Title compound"
        elif hyphenated:                             return "n_COMPOUND", "C2 hyphenated compound"
        elif bear_suffix:                            return "n_COMPOUND", "C3 ...bear compound"
        elif bear_adjective:                         return "n_COMPOUND", "C4 polar/grizzly/teddy/black/brown bear"
        # animal
        elif article_bear:                           return "n_ANIMAL", "N1 article/possessive + bear(s)"
        # symbolic, geography, agent
        elif symbol_next:                            return "v_SYMBOLICALLY", "S1 name/title/mark/resemblance... follows"
        elif direction_next:                         return "n_GEOGRAPHY", "G1 followed by a direction"
        elif resemblance:                            return "v_SYMBOLICALLY", "S2 bear a N to"
        elif bear_form in {"bearer", "bearers"}:          return "n_AGENT", "A1 bearer(s)"
        elif bear_form in {"bearing", "bearings"}:        return "n_GEOGRAPHY", "G2 noun bearing(s)"
        # animal (default)
        else:                                        return "n_ANIMAL", "N2 other noun uses"
    else:
        # idiom
        if bear_in_mind:                             return "v_IDIOM", "I1 bear in mind/remembrance"
        elif bear_fruit:                             return "v_IDIOM", "I2 bear fruit/witness/brunt/testimony"
        elif grin_and_bear:                          return "v_IDIOM", "I3 grin/smile and bear"
        elif bear_particle:                          return "v_IDIOM", "I4 bear down/away/out/up"
        # birth, symbolic
        elif child_next:                             return "v_GIVE_BIRTH", "B1 bear + child/son/..."
        elif symbol_next:                            return "v_SYMBOLICALLY", "S1 name/title/mark/resemblance... follows"
        # endure (infinitive), geography, carry (load word)
        elif infinitive_next or ing_next:            return "v_ENDURE", "E2 bear to + verb / bear + -ing"
        elif direction_next:                         return "n_GEOGRAPHY", "G1 followed by a direction"
        elif load_next:                              return "v_CARRY_PHYSICALLY", "K1 weight/load/box/... follows"
        # endure
        elif modal_before:                           return "v_ENDURE", "E1 after a modal or negative word"
        elif clause_end_next:                        return "v_ENDURE", "E3 bear at the end of a clause"
        elif pronoun_next:                           return "v_ENDURE", "E4 bear it/this/that"
        # carry (place), symbolic
        elif object_then_place:                      return "v_CARRY_PHYSICALLY", "K2 noun phrase + on/along/across/..."
        elif archaic:                                return "v_SYMBOLICALLY", "S3 old-fashioned wording in the sentence"
        elif resemblance:                            return "v_SYMBOLICALLY", "S2 bear a N to"
        # carry (default)
        else:                                        return "v_CARRY_PHYSICALLY", "K3 other verb uses"


BAG_AVAILABLE = "classify_rules" in globals()   # defined by Cell 5

if os.path.exists(dataset_filename):
    with open(dataset_filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    def new_bear_counts():
        return {"total": 0, "bag_match": 0, "syn_match": 0}

    bear_syntax_metrics = {cat: new_bear_counts() for cat in NOUN_BEARS + VERB_BEARS}
    noun_bear_syntax_totals, verb_bear_syntax_totals, grand_bear_syntax_totals = new_bear_counts(), new_bear_counts(), new_bear_counts()
    bear_rule_stats = defaultdict(lambda: {"category": "", "fired": 0, "right": 0})
    bear_misses = {cat: Counter() for cat in NOUN_BEARS + VERB_BEARS}

    for bear_spotted in data:
        words = bear_spotted.get("words", [])
        tags = bear_spotted.get("tags", [])
        bear_clue = bear_spotted.get("category", "UNCLASSIFIED")

        bear_id = next((k for k, w in enumerate(words) if "bear" in str(w).lower()), None)
        if bear_id is None:
            syn_bear_prediction, bear_rule_label = "UNCLASSIFIED", "no 'bear' token"
            bag_bear_prediction = "UNCLASSIFIED"
        else:
            target = str(words[bear_id]).lower()
            true_bear_pos = "VERB" if str(tags[bear_id]).startswith("V") else "NOUN"
            syn_bear_prediction, bear_rule_label = classify_bear_syntax(words, tags, bear_id, true_bear_pos)
            bag_bear_prediction = (classify_rules(" ".join([str(w) for w in words if w]), target, true_bear_pos)
                                   if BAG_AVAILABLE else None)

        if bear_clue in bear_syntax_metrics:
            bag_bear_ok = bag_bear_prediction == bear_clue
            syn_bear_ok = syn_bear_prediction == bear_clue
            for bear_bucket in (bear_syntax_metrics[bear_clue], grand_bear_syntax_totals, noun_bear_syntax_totals if bear_clue in NOUN_BEARS else verb_bear_syntax_totals):
                bear_bucket["total"] += 1
                bear_bucket["bag_match"] += bag_bear_ok
                bear_bucket["syn_match"] += syn_bear_ok
            bear_rule_stats[bear_rule_label]["category"] = syn_bear_prediction
            bear_rule_stats[bear_rule_label]["fired"] += 1
            bear_rule_stats[bear_rule_label]["right"] += syn_bear_ok
            bear_misses[bear_clue][syn_bear_prediction] += 1

    def print_bear_syntax_stat_row(label, bear_data_dict):
        total = bear_data_dict["total"]
        if total == 0:
            print(f"{label:<22} | {total:<5} | {'N/A':>8} | {'N/A':>8}")
            return
        bag_pct_val = f"{bear_data_dict['bag_match'] / total * 100:>7.1f}%" if BAG_AVAILABLE else f"{'N/A':>8}"
        syn_pct_val = f"{bear_data_dict['syn_match'] / total * 100:>7.1f}%"
        print(f"{label:<22} | {total:<5} | {bag_pct_val} | {syn_pct_val}")

    print(f"{'CATEGORY':<22} | {'COUNT':<5} | {'BAG %':>8} | {'SYNTAX %':>8}")
    print("=" * 52)

    for cat in NOUN_BEARS:
        print_bear_syntax_stat_row(cat, bear_syntax_metrics[cat])
    print("-" * 52)
    print_bear_syntax_stat_row("TOTAL NOUNS", noun_bear_syntax_totals)
    print("=" * 52)

    for cat in VERB_BEARS:
        print_bear_syntax_stat_row(cat, bear_syntax_metrics[cat])
    print("-" * 52)
    print_bear_syntax_stat_row("TOTAL VERBS", verb_bear_syntax_totals)
    print("=" * 52)

    print_bear_syntax_stat_row("GRAND TOTAL", grand_bear_syntax_totals)
    print("=" * 52)

    if SHOW_RULE_STATS:
        bear_rule_order = ["C1", "C2", "C3", "C4", "N1", "I1", "I2", "I3", "I4", "B1", "S1", "E2", "G1", "K1", "E1", "E3", "E4", "K2", "S3", "S2", "A1", "G2", "N2", "K3"]
        print(f"\n{'RULE':<42} | {'PREDICTS':<20} | {'FIRED':>5} | {'RIGHT %':>8}")
        print("-" * 84)
        for label in sorted(bear_rule_stats, key=lambda l: bear_rule_order.index(l.split()[0]) if l.split()[0] in bear_rule_order else 99):
            s = bear_rule_stats[label]
            print(f"{label:<42} | {s['category']:<20} | {s['fired']:>5} | {s['right'] / s['fired'] * 100:>7.1f}%")

    if SHOW_BEAR_MISSES:
        print()
        for cat in NOUN_BEARS + VERB_BEARS:
            if bear_misses[cat]:
                total = sum(bear_misses[cat].values())
                where = ", ".join(f"{c} {k}" for c, k in bear_misses[cat].most_common(4))
                print(f"{cat} ({total}): {where}")

else:
    print(f"Dataset not found at {dataset_filename}")

CATEGORY               | COUNT |    BAG % | SYNTAX %
n_ANIMAL               | 46    |    82.6% |    84.8%
n_AGENT                | 27    |    55.6% |    85.2%
n_GEOGRAPHY            | 7     |    71.4% |    85.7%
n_COMPOUND             | 144   |    89.6% |    89.6%
----------------------------------------------------
TOTAL NOUNS            | 224   |    83.5% |    87.9%
v_CARRY_PHYSICALLY     | 109   |    46.8% |    48.6%
v_ENDURE               | 193   |    18.7% |    65.3%
v_GIVE_BIRTH           | 46    |    17.4% |    23.9%
v_SYMBOLICALLY         | 115   |    14.8% |    40.0%
v_IDIOM                | 95    |    31.6% |    31.6%
----------------------------------------------------
TOTAL VERBS            | 558   |    25.4% |    47.7%
GRAND TOTAL            | 782   |    42.1% |    59.2%


In [21]:
# Tested Gemma on 50 Human-Annotated Sentences
human_labelled_bears = [bear_spotted for bear_spotted in load_dataset() if bear_spotted.get("human_category")]
correct_bear_predictions = sum(
    bear_spotted["human_category"].strip().lower() in bear_spotted["category"].lower() 
    for bear_spotted in human_labelled_bears
)
total_labelled_bears = max(len(human_labelled_bears), 1)
accuracy_percentage = correct_bear_predictions / total_labelled_bears

print(f"Gemma's hunting accuracy: {correct_bear_predictions} of {len(human_labelled_bears)} ({accuracy_percentage:.1%})")

Gemma's hunting accuracy: 45 of 50 (90.0%)
